# 01 · XLS-R layer sweep (Stage 2)

Reads the outputs of `scripts/layer_sweep.py` and `scripts/verify_truncation.py`; it does not recompute anything.

**DEV speakers only.** Test speakers (`corpus.test_speakers` in `config/settings.yaml`) are never used to choose the layer; `src/splits.py` enforces this.

Task: does a child's word token have a lower DTW cost to the *same word* from other children than to *different* words? Metrics: ROC-AUC over all pairs and P@1 per query.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("..") / "eval" / "results"
sweep = pd.read_csv(RESULTS / "layer_sweep_dev.csv")
print((RESULTS / "layer_sweep_dev.md").read_text(encoding="utf-8").split("|")[0])
sweep.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for col, label in [("auc", "AUC (all)"), ("auc_en", "AUC English"), ("auc_si", "AUC Sinhala"), ("p_at_1", "P@1")]:
    ax.plot(sweep.layer, sweep[col], marker="o", ms=3, label=label)
best = sweep.loc[sweep.auc.idxmax()]
ax.axvline(best.layer, color="grey", ls="--", lw=1)
ax.set_xlabel("XLS-R layer"); ax.set_ylabel("score (dev speakers)"); ax.grid(alpha=0.3); ax.legend()
ax.set_title(f"Best layer by AUC: {int(best.layer)}")

## Truncated encoder check

Keeping `layer + 1` transformer layers must reproduce the full model's `hidden_states[layer]` exactly. The naive cut to `layer` layers is shown as a control (it should *not* match, because XLS-R applies its final layer norm only to the last output).

In [ ]:
check = json.loads((RESULTS / "truncation_check.json").read_text())
pd.DataFrame(check["results"])